# FinDPO reproduction — analysis walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khoaminh2957/findpo/blob/main/notebooks/findpo_reproduction.ipynb)

**Paper:** Iacovides, Zhou & Mandic 2025 — [arXiv:2507.18417](https://arxiv.org/abs/2507.18417) — *"FinDPO: Financial Sentiment Classification with Direct Preference Optimization"*

**Repo:** https://github.com/khoaminh2957/findpo

---

## What this notebook shows

This is an **analysis** notebook — it does NOT re-train the models (training takes ~7h on an A100, far beyond Colab limits). It loads the committed eval JSONs and raw predictions from the reproduction and walks through the key findings:

1. **Phase 1 SFT** — exceeds paper's FinSFT baseline on all 3 datasets ✅
2. **Phase 2 DPO** — regresses vs SFT (opposite direction from paper) ❌
3. **Per-example analysis** — DPO loses ~2× as many examples as it gains, on every seed
4. **Output format degradation** — DPO model emits noisy tokens (`positive▍`, `negativeassistant`, …)
5. **Per-checkpoint test** — rules out over-training as sole cause; DPO is worse than SFT at every saved epoch

The full failure analysis with root-cause hypotheses lives in [`reports/REPRODUCTION_REPORT.md`](https://github.com/khoaminh2957/findpo/blob/main/reports/REPRODUCTION_REPORT.md).

## 0. Setup — clone the repo, install minimal deps

No GPU needed; this section just downloads the committed result files.

In [ ]:
!git clone --depth 1 https://github.com/khoaminh2957/findpo.git
%cd findpo
!git log --oneline -5

In [ ]:
# Only need stdlib + matplotlib for this analysis notebook.
import json, glob, os
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('.')
SEEDS = [42, 123, 7]
DATASETS = ['fpb', 'tfns', 'gpt_news']
DATASET_LABEL = {'fpb': 'FPB', 'tfns': 'TFNS', 'gpt_news': 'NWGI'}
print(f'cwd = {ROOT.resolve()}')
print(f'seeds = {SEEDS}, datasets = {DATASETS}')

## 1. Phase 1 — SFT baseline

Llama-3.1-8B-Instruct + QLoRA NF4 (r=16, α=16, lr=2e-4, 3 epochs).

The paper reports FinSFT weighted F1 of 0.829 (FPB) / 0.850 (TFNS) / 0.708 (NWGI). Let's load our eval JSONs and see how we did.

In [ ]:
def load_eval(exp_id, seed, dataset):
    p = ROOT / 'results' / f'exp_{exp_id}_seed{seed}' / f'eval_{dataset}.json'
    if not p.exists():
        return None
    return json.loads(p.read_text())

def mean_std(values):
    a = np.array(values, dtype=float)
    return a.mean(), a.std(ddof=1)

# Paper's FinSFT and FinDPO numbers from Table 2.
PAPER = {
    'sft': {'fpb': 0.829, 'tfns': 0.850, 'gpt_news': 0.708},
    'dpo': {'fpb': 0.865, 'tfns': 0.872, 'gpt_news': 0.833},
}

rows = []
for ds in DATASETS:
    wf1s = [load_eval('sft_baseline', s, ds)['weighted_f1'] for s in SEEDS]
    mu, sd = mean_std(wf1s)
    rows.append((DATASET_LABEL[ds], mu, sd, PAPER['sft'][ds], mu - PAPER['sft'][ds]))

print(f'{"Dataset":<6} {"Repro SFT (w-F1)":<22} {"Paper FinSFT":<14} {"Δ":<10}')
print('-' * 60)
for name, mu, sd, paper, delta in rows:
    sign = '+' if delta >= 0 else ''
    print(f'{name:<6} {mu:.4f} ± {sd:.4f}        {paper:.3f}          {sign}{delta:.3f}')
print()
mean_repro = np.mean([r[1] for r in rows])
mean_paper = np.mean([r[3] for r in rows])
print(f'Mean:  repro {mean_repro:.4f}   paper {mean_paper:.4f}   Δ +{mean_repro - mean_paper:.3f}')

**Result:** Our SFT exceeds the paper's FinSFT baseline by ~0.09 weighted F1 on average. This is the strongest single piece of evidence that the pipeline (dataset loading, prompt formatting, tokenisation, chat template, label canonicalisation, eval code) is **correct**.

→ Any failure in Phase 2 cannot be blamed on a pipeline bug. Keep this in mind below.

## 2. Phase 2 — DPO (the headline)

Same base model + LoRA. DPO trained on Strategy AB preference pairs:
- If SFT predicts correctly → rejected = random non-true class
- If SFT predicts wrong → rejected = SFT's predicted (wrong) class
- Chosen = always the true label

TRL `DPOTrainer` 0.12.2, `ref_model=None` + `peft_config` (i.e., reference = base model with LoRA disabled). 5 epochs, lr 5e-6, β 0.1.

Paper Table 2 implies +6.1 pp simple-mean improvement of FinDPO over FinSFT (averaging FPB +3.6, TFNS +2.2, NWGI +12.5). Let's see what we got.

In [ ]:
rows = []
for ds in DATASETS:
    sft_wf1s = [load_eval('sft_baseline', s, ds)['weighted_f1'] for s in SEEDS]
    dpo_wf1s = [load_eval('dpo_repro_qlora', s, ds)['weighted_f1'] for s in SEEDS]
    sft_mu, sft_sd = mean_std(sft_wf1s)
    dpo_mu, dpo_sd = mean_std(dpo_wf1s)
    delta = dpo_mu - sft_mu
    paper_delta = PAPER['dpo'][ds] - PAPER['sft'][ds]
    rows.append((DATASET_LABEL[ds], sft_mu, sft_sd, dpo_mu, dpo_sd, delta, paper_delta))

print(f'{"Dataset":<6} {"SFT (w-F1)":<18} {"DPO (w-F1)":<18} {"Δ DPO-SFT":<10} {"Paper Δ":<10} {"Verdict":<12}')
print('-' * 80)
for name, sm, ss, dm, ds_, delta, pdelta in rows:
    sign = '+' if delta >= 0 else ''
    psign = '+' if pdelta >= 0 else ''
    verdict = '✓ matches' if abs(delta - pdelta) < 0.02 else ('✗ regress' if delta < 0 else '~ partial')
    print(f'{name:<6} {sm:.4f} ± {ss:.4f}   {dm:.4f} ± {ds_:.4f}   {sign}{delta:.3f}    {psign}{pdelta:.3f}    {verdict}')
print()
mean_delta = np.mean([r[5] for r in rows])
mean_pdelta = np.mean([r[6] for r in rows])
print(f'Mean Δ DPO-SFT:  repro {mean_delta:+.3f}   paper {mean_pdelta:+.3f}')

**Result:** DPO regresses on every dataset and every seed. Average regression is −3.5 pp weighted F1, vs the +6.1 pp simple-mean improvement implied by paper Table 2 — a ~10 pp gap.

Let's visualise this side-by-side with the paper's numbers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(DATASETS))
width = 0.22

sft_means = [mean_std([load_eval('sft_baseline', s, ds)['weighted_f1'] for s in SEEDS])[0] for ds in DATASETS]
sft_stds  = [mean_std([load_eval('sft_baseline', s, ds)['weighted_f1'] for s in SEEDS])[1] for ds in DATASETS]
dpo_means = [mean_std([load_eval('dpo_repro_qlora', s, ds)['weighted_f1'] for s in SEEDS])[0] for ds in DATASETS]
dpo_stds  = [mean_std([load_eval('dpo_repro_qlora', s, ds)['weighted_f1'] for s in SEEDS])[1] for ds in DATASETS]
paper_sft  = [PAPER['sft'][ds] for ds in DATASETS]
paper_dpo  = [PAPER['dpo'][ds] for ds in DATASETS]

ax.bar(x - 1.5*width, sft_means, width, yerr=sft_stds, label='Repro SFT', color='#3b82f6', capsize=4)
ax.bar(x - 0.5*width, dpo_means, width, yerr=dpo_stds, label='Repro DPO', color='#ef4444', capsize=4)
ax.bar(x + 0.5*width, paper_sft, width, label='Paper FinSFT', color='#93c5fd', alpha=0.7)
ax.bar(x + 1.5*width, paper_dpo, width, label='Paper FinDPO', color='#fca5a5', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels([DATASET_LABEL[d] for d in DATASETS])
ax.set_ylabel('Weighted F1')
ax.set_ylim(0.65, 0.95)
ax.set_title('Reproduction vs paper — weighted F1 (3-seed mean ± std)')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Notice two things:
1. **Our SFT (blue solid) sits ABOVE the paper's FinSFT (blue faded)** on every dataset — pipeline is fine.
2. **Our DPO (red solid) sits BELOW our SFT** on every dataset — DPO regressed.
3. **The paper's FinDPO (red faded) is ABOVE its FinSFT** — that's the +6.1 pp simple-mean improvement we failed to reproduce.

## 3. Per-example analysis — where does the loss come from?

Aggregate F1 numbers hide *which examples* DPO got wrong. Both the SFT and DPO eval pipelines save raw per-example predictions to `predictions_<dataset>.jsonl`. We can join them by input text and ask:
- How many examples did **both** SFT and DPO get right?
- How many did SFT get right and DPO get wrong? (DPO regression)
- How many did DPO fix that SFT missed? (DPO improvement)

If DPO is genuinely helpful, the second column should be larger than the third. The opposite would mean DPO is destroying SFT's correct answers faster than it fixes new ones.

In [ ]:
def load_preds(exp_id, seed, dataset):
    p = ROOT / 'results' / f'exp_{exp_id}_seed{seed}' / f'predictions_{dataset}.jsonl'
    if not p.exists():
        return None
    return {ex['text']: ex for ex in (json.loads(l) for l in p.open(encoding='utf-8'))}

def compare_seed(seed, dataset='fpb'):
    sft = load_preds('sft_baseline', seed, dataset)
    dpo = load_preds('dpo_repro_qlora', seed, dataset)
    common = set(sft) & set(dpo)
    both = sft_only = dpo_only = both_wrong = 0
    for t in common:
        sok = sft[t]['pred'] == sft[t]['true']
        dok = dpo[t]['pred'] == dpo[t]['true']
        if sok and dok: both += 1
        elif sok and not dok: sft_only += 1
        elif not sok and dok: dpo_only += 1
        else: both_wrong += 1
    return both, sft_only, dpo_only, both_wrong, len(common)

print(f'FPB test set — SFT predictions vs DPO predictions, joined by input text')
print(f'{"Seed":<6} {"Both right":<12} {"SFT only":<12} {"DPO only":<12} {"Both wrong":<12} {"Net DPO":<10}')
print('-' * 72)
for s in SEEDS:
    br, so, do_, bw, n = compare_seed(s, 'fpb')
    print(f'{s:<6} {br:<12} {so:<12} {do_:<12} {bw:<12} {do_ - so:+}')
print()
print('"SFT only" = SFT was right, DPO became wrong  →  DPO regression')
print('"DPO only" = DPO fixed an example SFT missed  →  DPO improvement')
print('Net DPO  = DPO only - SFT only.  Negative = DPO is destroying more than it fixes.')

Confirmed across all 3 seeds: DPO **loses ~2× more examples than it gains**. This is a structural failure, not seed-specific noise.

What kind of errors does DPO introduce? Let's bucket the regressions by `(true label → DPO wrong prediction)`:

In [ ]:
def regression_breakdown(seed, dataset='fpb'):
    sft = load_preds('sft_baseline', seed, dataset)
    dpo = load_preds('dpo_repro_qlora', seed, dataset)
    c = Counter()
    for t in set(sft) & set(dpo):
        if sft[t]['pred'] == sft[t]['true'] and dpo[t]['pred'] != dpo[t]['true']:
            c[(sft[t]['true'], dpo[t]['pred'])] += 1
    return c

print('SEED 42, FPB — error transitions (true → DPO-wrong prediction) where SFT was right')
for (true, dpo_pred), n in sorted(regression_breakdown(42, 'fpb').items(), key=lambda kv: -kv[1]):
    print(f'  {true:<10} → {dpo_pred:<10}  {n}')

**Bidirectional drift toward neutral:**
- DPO over-neutralises confident `positive` / `negative` SFT predictions (~42 cases)
- DPO simultaneously misclassifies truly-neutral inputs as `positive` or `negative` (~26 cases)

This is **not** a simple "DPO collapsed to predicting neutral". The decision boundary shifted in a way that hurts both directions. The likely cause is Strategy AB's random-class rejecteds (when SFT was already correct, the rejected is a random non-true label) creating noisy gradient that pushes the model away from confident predictions in either direction.

## 4. Output format degradation

Beyond classification accuracy, the DPO model also produces noisier raw outputs. The eval script generates up to 4 new tokens and parses the first label found. Let's see the distribution of unique raw outputs:

In [ ]:
def raw_distribution(exp_id, seed, dataset):
    preds = load_preds(exp_id, seed, dataset)
    return Counter(p['raw'] for p in preds.values())

for exp, label in [('sft_baseline', 'SFT'), ('dpo_repro_qlora', 'DPO')]:
    c = raw_distribution(exp, 42, 'fpb')
    print(f'\n=== {label} seed=42 FPB — {len(c)} unique raw outputs ===')
    for raw, n in c.most_common(8):
        # Pad/quote for readability
        print(f'  {n:>4}  {raw!r}')

**SFT:** a handful of clean unique outputs (`'positive'`, `'neutral'`, `'negative'` and tiny variants).

**DPO:** dozens of unique outputs including:
- `'positive▍'` — Unicode box-drawing glyph (U+258F) leaks into the token stream
- `'negativeassistant'` — model emits the chat role token verbatim
- `' neutral (This is'`, `'neutral//**\nThere is'` — verbose narrative that would be truncated at `max_new_tokens=4`
- `'positive)))),'` — repetitive punctuation noise

Most still parse correctly because `_parse_label()` does prefix matching. But this is a clear signal of distribution drift — DPO has shifted the model's output distribution AWAY from the clean single-word format that SFT learned.

`n_unparseable` across 9 evals: SFT total = 0 / 17,364, DPO total = 157 / 17,364.

## 5. Per-checkpoint test — is it just over-training?

`eval_loss` rises after epoch 2-3 in the train log — classic over-training. But is over-training the whole story?

We re-evaluated each preserved checkpoint of seed 42 on FPB. (Checkpoints 2750 and 5500 were deleted mid-train to free disk on a 32 GB Vast.ai container — see the report's §6 procedural note.)

| Checkpoint | Epoch | Weighted F1 | Unparseable |
|---|---|---|---|
| ckpt-8250  | 3 (earliest preserved) | 0.8396 | 0 |
| ckpt-11000 | 4 ("best" by `eval_rewards/accuracies`) | 0.8500 | 3 |
| ckpt-13750 | 5 (final) | 0.8484 | 3 |
| **SFT baseline (epoch 3)** | **—** | **0.8891** | **0** |

Even the earliest preserved DPO checkpoint is **0.05 weighted F1 below SFT**. Over-training amplifies the format-degradation symptom (unparseable goes 0 → 3), but the classification gap is already there at epoch 3. Earlier epochs would have to lift weighted F1 by 0.05 in just 1-2 fewer epochs — unlikely, given that `eval_rewards/accuracies` (the DPO metric) plateaus by epoch 2.

**Conclusion:** over-training contributes but is not the sole cause. The DPO setup itself is producing a model that classifies worse than its own initialisation.

## 6. The DPO training metric and the downstream metric diverged

From the W&B logs (also in `results/exp_dpo_repro_qlora_seed*/train_log.jsonl`):

| Seed | Best `eval_rewards/accuracies` | At epoch |
|---|---|---|
| 42 | 0.9148 | 4 |
| 123 | 0.9168 | 2 |
| 7 | 0.9225 | 3 |

On held-out preference pairs, the DPO model correctly ranks chosen vs rejected for **>91 %** of pairs. By the DPO metric, training is succeeding.

But the downstream classification accuracy on the same test inputs is **worse than SFT**. The two metrics have decorrelated.

Why? `eval_rewards/accuracies` measures "does the policy assign a higher log-prob to the chosen response than to the rejected response, relative to the reference model?". Maximising this is **not** the same as "correctly classify the input". When the reference model is the base Llama (with LoRA disabled, so it cannot classify), the DPO loss is mostly rewarding any change to the policy that boosts the chosen tokens — even changes that hurt overall calibration.

Combined with Strategy AB's random-class rejecteds (88 % noise), this gives the model a strong but mis-aimed gradient. The model satisfies the DPO objective by drifting away from the SFT-quality initialisation in directions that don't help — and often hurt — actual classification.

## 7. Root-cause hypotheses (untested)

Most-cheap-to-test first:

1. **Reference model should be the SFT model, not the base.** TRL's Zephyr-style `ref_model=None` + `peft_config` makes the reference the base Llama with LoRA disabled. With ref = SFT, β = 0.1 anchors the DPO drift to the SFT-quality initialisation. Single seed, 2-3 epochs would be enough to test.
2. **Strategy A only.** Restrict pairs to the subset where SFT was actually wrong (chosen = truth, rejected = SFT's confused class). Removes the random-class noise that dominates the training signal.
3. **Earlier stopping / fewer epochs.** `eval_loss` bottoms at epoch 2-3. Either run 2-3 epochs or use `eval_loss` (not `eval_rewards/accuracies`) as `metric_for_best_model`.
4. **Higher β.** Currently 0.1. Combined with hypothesis 1, lower drift would protect the SFT initialisation.
5. **Llama-3.0 vs 3.1.** Paper used 3.0 (gated); we used the architecturally identical 3.1 via the `unsloth` mirror.

See [`reports/REPRODUCTION_REPORT.md`](https://github.com/khoaminh2957/findpo/blob/main/reports/REPRODUCTION_REPORT.md) for the full discussion.

## 8. Inspect specific examples (interactive)

Pick any FPB test example and see what SFT predicted, what DPO predicted, and the raw model outputs:

In [ ]:
# Show 5 examples where SFT was right and DPO became wrong (seed 42)
sft42 = load_preds('sft_baseline', 42, 'fpb')
dpo42 = load_preds('dpo_repro_qlora', 42, 'fpb')

regressions = [t for t in sft42 if t in dpo42 and sft42[t]['pred'] == sft42[t]['true'] and dpo42[t]['pred'] != dpo42[t]['true']]
print(f'Total regressions on seed 42 FPB: {len(regressions)}\n')

for t in regressions[:5]:
    s, d = sft42[t], dpo42[t]
    text = t[:90] + ('…' if len(t) > 90 else '')
    print(f'TRUE:     {s["true"]}')
    print(f'SFT pred: {s["pred"]:<10}  raw={s["raw"]!r}')
    print(f'DPO pred: {d["pred"]:<10}  raw={d["raw"]!r}')
    print(f'INPUT:    {text}')
    print('-' * 80)

## 9. Reproducing the numbers in this notebook

Everything shown here was computed from files committed to the repo at tag `phase-4-complete`:

- `results/exp_sft_baseline_seed{42,123,7}/eval_{fpb,tfns,gpt_news}.json` — 9 files (used by §1, §2, §5)
- `results/exp_dpo_repro_qlora_seed{42,123,7}/eval_{fpb,tfns,gpt_news}.json` — 9 files (used by §1, §2, §5)
- `results/exp_dpo_repro_qlora_seed{42,123,7}/predictions_{fpb,tfns,gpt_news}.jsonl` — 9 files (used by §3, §4, §8)
- `results/exp_sft_baseline_seed{42,123,7}/predictions_fpb.jsonl` — 3 files (FPB only — needed for the §3 / §4 / §8 SFT⋈DPO join)

If you want to re-run the training itself (not in Colab — needs a real GPU for ~7 h):

```bash
# Phase 0
python scripts/00_download_datasets.py --config configs/sft_baseline.yaml

# Phase 1 (one seed per GPU, 3 seeds in parallel)
for s in 42 123 7; do
  CUDA_VISIBLE_DEVICES=$((s%3)) python scripts/01_sft_train.py --config configs/sft_baseline.yaml --seed $s &
done; wait

# Phase 2 — pref pairs + DPO + eval
for s in 42 123 7; do
  python scripts/02_build_preference_pairs.py --config configs/dpo_repro_qlora.yaml \
    --strategy AB --seed $s --sft-run-dir results/exp_sft_baseline_seed$s
done

for s in 42 123 7; do
  CUDA_VISIBLE_DEVICES=$((s%3)) python scripts/02_dpo_train.py \
    --config configs/dpo_repro_qlora.yaml --seed $s \
    --pairs-dir data/preference_pairs/strategyAB_seed$s &
done; wait
```

See [`README.md`](https://github.com/khoaminh2957/findpo/blob/main/README.md) for full setup (Vast.ai Docker template, pip-first install, flash-attn notes, etc.).

> Note: re-running the SFT eval on a different hardware/software stack (e.g., A100+flash-attn vs RTX 5090+sdpa) can introduce tiny numerical differences (~10⁻³ in weighted F1) because the generation path through different attention kernels is not bit-identical. The original Phase 1 SFT eval was on 5090+sdpa; FPB was re-evaluated on A100+sdpa to obtain `predictions_fpb.jsonl` for the §3 join. The 3-seed SFT FPB mean moved from 0.8898 ± 0.0055 → 0.8898 ± 0.0045 — within 0.001 noise and does not affect any conclusion.

## Closing thought

Reproducing a published result that does not reproduce is uncomfortable — but a clean negative result with isolated root-cause hypotheses is more useful to the field than a vague success or silent failure. The SFT phase matched the paper; the DPO phase did not. The most likely culprit (per §7) is the reference-model choice, which the paper does not specify explicitly. A future run with `ref_model = SFT` is the next experiment to do.